# GPT 생성과 검색 수 확대

사업 제한 없이 후보 20개씩 검색하고 최종 근거 6개를 사용한 실험. 다른 사업 근거 혼입을 확인했습니다.

실행 출력은 GitHub 공유를 위해 제거했습니다. 원본 문서와 평가 데이터는 별도로 준비해야 합니다.


## 준비. 패키지 설치와 드라이브 연결


In [ ]:
%pip install -q numpy pandas faiss-cpu "sentence-transformers==5.6.0" langchain-huggingface langchain-community langchain-text-splitters pymupdf4llm langchain-hwp-hwpx-loader "transformers==4.51.3" "autoawq==0.2.9" accelerate rank_bm25
%pip install -U "ragas==0.3.9" "langchain-openai"


In [ ]:
import os
import re
import unicodedata

import pandas as pd
import pymupdf4llm
from google.colab import drive
from langchain_core.documents import Document
from langchain_hwp_hwpx import HwpHwpxLoader
from hwp_hwpx_parser import ExtractOptions, TableStyle
from langchain_text_splitters import RecursiveCharacterTextSplitter

drive.mount("/content/drive")
folder = "/content/drive/MyDrive/data"
metadata_csv = "/content/drive/MyDrive/data_list.csv"


In [ ]:
# 문서 순서를 고정해 다시 실행해도 청크 번호가 같은 순서로 생성되도록 합니다.
sorted_file_names = sorted(
    os.listdir(folder),
    key=lambda name: (unicodedata.normalize("NFKC", name).casefold(), name),
)
print("정렬한 파일 수:", len(sorted_file_names))

## 1. 문서 로드 (HWP, PDF)


In [ ]:
def normalize_filename(name):
    name = str(name).rsplit(".", 1)[0]
    name = unicodedata.normalize("NFKC", name)
    return re.sub(r"\s+", " ", name).strip()


df = pd.read_csv(metadata_csv, encoding="utf-8")
loaded_files = []

for name in sorted_file_names:
    full_path = os.path.join(folder, name)

    extension = name.rsplit(".", 1)[-1].lower()
    if extension == "pdf":
        text = pymupdf4llm.to_markdown(full_path)
    elif extension in ("hwp"):
        options = ExtractOptions(table_style=TableStyle.MARKDOWN)
        loader = HwpHwpxLoader(full_path, mode="single", extract_options=options)
        text = loader.load()[0].page_content

    loaded_files.append((name, text))

print("로드한 문서:", len(loaded_files))


## 2. 메타데이터와 본문을 Document로 저장

`page_content`에 본문을, `metadata`에 CSV의 문서 정보를 저장합니다.


In [ ]:
metadata_columns = [
    "공고 번호", "공고 차수", "사업명", "사업 금액", "발주 기관",
    "공개 일자", "입찰 참여 시작일", "입찰 참여 마감일", "파일명",
]

documents = []
for name, text in loaded_files:
    result_df = df[df["파일명"].apply(normalize_filename) == normalize_filename(name)]
    if result_df.empty:
        continue

    row = result_df.iloc[0]
    metadata = {column: row[column] for column in metadata_columns}
    documents.append(Document(page_content=text, metadata=metadata))

print("메타데이터와 연결한 문서:", len(documents))


## 3. 본문 기본 전처리

연속 공백과 빈 줄만 정리합니다.


In [ ]:
for document in documents:
    text = document.page_content.replace("\r\n", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    document.page_content = re.sub(r"\n{3,}", "\n\n", text).strip()


## 4. 청킹


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " "], chunk_size=700, chunk_overlap=100
)

chunks = splitter.split_documents(documents)
for chunk_id, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = chunk_id

print("청크 수:", len(chunks))


## 5. 임베딩 후 FAISS DB에 저장

`embedding_model_name`을 바꾸면 임베딩 모델을 조정할 수 있습니다.


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model_name = "Alibaba-NLP/gte-multilingual-base"
embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={"device": "cuda", "trust_remote_code": True},
    encode_kwargs={"batch_size": 64, "normalize_embeddings": True},
)

vector_store = FAISS.from_documents(chunks, embeddings)
print("저장한 청크:", vector_store.index.ntotal)


## 6. 평가 질문 불러오기

`rag_eval_dataset.json`을 Google Drive의 `내 드라이브` 루트에 올려 두고 실행합니다. 로컬에서 실행할 때는 노트북과 같은 폴더에 둡니다. 전체 180문항을 사용하며, 필요한 경우 아래 셀의 `EVAL_PROJECTS` 또는 `EVAL_LIMIT`으로 범위를 줄일 수 있습니다.


In [ ]:
import json
from collections import Counter
from pathlib import Path

eval_dataset_candidates = [
    Path("/content/drive/MyDrive/rag_eval_dataset.json"),
    Path("rag_eval_dataset.json"),
]
eval_dataset_path = next((path for path in eval_dataset_candidates if path.is_file()), None)
if eval_dataset_path is None:
    raise FileNotFoundError(
        "rag_eval_dataset.json을 Google Drive 내 드라이브 루트 또는 노트북 폴더에 두세요."
    )

with eval_dataset_path.open(encoding="utf-8") as file:
    all_eval_records = json.load(file)

EVAL_PROJECTS = None  # 예: {"수문자료정보관리시스템(HDIMS) 재구축사업(3단계)"}
EVAL_LIMIT = 12     # 빠른 시범 실행 시 예: 12. None이면 전부 사용합니다.

selected_eval_records = [
    record for record in all_eval_records
    if EVAL_PROJECTS is None or record["project"] in EVAL_PROJECTS
]
if EVAL_LIMIT is not None:
    selected_eval_records = selected_eval_records[:EVAL_LIMIT]
if not selected_eval_records:
    raise ValueError("선택된 평가 문항이 없습니다. 사업명 필터와 입력 파일을 확인하세요.")

evaluation_data = []
for record in selected_eval_records:
    if not record.get("id") or not record.get("question") or not record.get("reference"):
        raise ValueError(f"필수 평가 필드가 비어 있습니다: {record.get('id')}")
    evaluation_data.append({
        "id": record["id"],
        "query": record["question"],
        "reference": record["reference"],
        "project": record["project"],
        "question_type": record["question_type"],
    })

if len({item["id"] for item in evaluation_data}) != len(evaluation_data):
    raise ValueError("평가 문항 ID가 중복됩니다.")
if len({item["query"] for item in evaluation_data}) != len(evaluation_data):
    raise ValueError("평가 질문이 중복됩니다.")

question = evaluation_data  # 기존 검색·답변 생성 반복문에서 사용
print(f"평가셋: {eval_dataset_path} / {len(evaluation_data)}문항")
print("유형별:", dict(Counter(item["question_type"] for item in evaluation_data)))
print("사업별:", dict(Counter(item["project"] for item in evaluation_data)))


## 7. 하이브리드 검색 (Dense + BM25, RRF)


In [ ]:
from langchain_community.retrievers import BM25Retriever

dense_retriever = vector_store.as_retriever(search_kwargs={"k": 20})
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 20


def rrf_search(query, k=60):
    dense_results = dense_retriever.invoke(query)
    sparse_results = bm25_retriever.invoke(query)
    scores = {}

    for results in (dense_results, sparse_results):
        for rank, doc in enumerate(results, start=1):
            doc_id = doc.metadata["chunk_id"]
            if doc_id not in scores:
                scores[doc_id] = {"doc": doc, "score": 0}
            scores[doc_id]["score"] += 1 / (k + rank)

    ranked = sorted(scores.values(), key=lambda item: item["score"], reverse=True)
    return [item["doc"] for item in ranked]


## 8. HyDE로 검색 문장 만들기


In [ ]:
import gc
import torch

"del embeddings"

gc.collect()
torch.cuda.empty_cache()

In [ ]:
hyde_search_rules = '''너는 RFP에서 관련 문단을 찾기 위한 가상 안내문 작성자야.
실제 답변이 아니라 검색에 쓸 안내문 1문장만 작성해.
- 질문의 핵심 주제어를 그대로 포함해.
- 질문과 관계없는 다른 사업이나 항목으로 이동하지 마.
- 확인되지 않은 금액, 날짜, 법률, 자격 조건을 단정하지 마.
- 제목, 목록, 따옴표, 추가 설명을 쓰지 마.

예시:
질문: 이 사업의 입찰 참여 자격은 무엇인가요?
가상 안내문: 이 사업의 입찰 참여 자격에 관한 조건이 안내되어 있다.'''

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace


language_model_name = "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct-AWQ"
model_revision = "f2699f35d5b4ba511ab93826c22bd07370296b5c"


model = AutoModelForCausalLM.from_pretrained(
    language_model_name, torch_dtype=torch.float16,
    trust_remote_code=True, device_map="auto", revision=model_revision
)
tokenizer = AutoTokenizer.from_pretrained(language_model_name, revision=model_revision)


llm_pipeline = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    do_sample=False,
    repetition_penalty=1.2,
    return_full_text=False,
    max_new_tokens=220,
)


llm = HuggingFacePipeline(pipeline=llm_pipeline)
chat_model = ChatHuggingFace(llm=llm)

method = "basic"  # "basic" 또는 "hyde"


In [ ]:
import os
from langchain_openai import ChatOpenAI

method = "basic"  # "basic" 또는 "hyde"

# Colab 보안 비밀에 키를 등록한 경우
if not os.getenv("OPENAI_API_KEY"):
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

chat_model = ChatOpenAI(model="gpt-5-mini-2025-08-07")

In [ ]:
def make_search_text(query, method):
    if method == "basic":
        return query
    if method == "hyde":
        hyde_prompt = f"{hyde_search_rules}\n\n질문: {query}\n가상 안내문:"
        return chat_model.invoke(hyde_prompt).content.strip()
    raise ValueError(f"지원하지 않는 검색 방식: {method}")

## 9. 재순위화


In [ ]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "Alibaba-NLP/gte-multilingual-reranker-base",
    trust_remote_code=True,
    device="cuda"
)

# 확인 셀과 답변 생성 셀이 같은 검색·재순위화 결과를 공유합니다.
retrieval_cache = {}


def retrieve_and_rerank(query, method):
    cache_key = (method, query)
    if cache_key not in retrieval_cache:
        search_text = make_search_text(query, method)
        found = rrf_search(search_text)
        ranking = reranker.rank(
            query,  # 재순위화에는 원래 질문 사용
            [doc.page_content for doc in found],
            top_k=6,
        )
        reranked_docs = [found[row["corpus_id"]] for row in ranking]
        retrieval_cache[cache_key] = (search_text, found, reranked_docs)
    return retrieval_cache[cache_key]


for i, q in enumerate(question, start=1):
    query = q["query"]
    search_text, found, reranked_docs = retrieve_and_rerank(query, method)
    print(f"\n{i}번째 질문: {query}")
    print("검색 문장:", search_text)
    print("리랭크 전:", [doc.metadata["chunk_id"] for doc in found])
    print("리랭크 후:", [doc.metadata["chunk_id"] for doc in reranked_docs])


## 10. 답변 생성


In [ ]:
import gc
import torch

"del embeddings"

gc.collect()
torch.cuda.empty_cache()

In [ ]:
# 위 셀의 검색·재순위화 결과로 최종 답변을 한 번만 생성합니다.
answer_rules = '''
너는 RFP 문서의 사실 확인 담당자다. 아래 <문서>에 적힌 내용만으로 <질문>에 답한다.

규칙:
1. 질문이 여러 항목을 묻는다면 항목별로 근거를 따로 찾는다.
2. 질문의 사업명과 다른 사업의 정보는 사용하지 않는다.
3. 각 항목에 대해 문서에서 직접 인용할 수 있는 문구를 먼저 제시한다.
4. 직접 인용할 근거가 없는 항목은 추측하지 말고 "제공된 문서에서 확인할 수 없습니다"라고 쓴다.
5. 금액, 기간, 날짜, 비율, 인력, 제출 조건은 문서의 표현을 그대로 옮긴다. 계산하거나 일반적인 관행으로 빈칸을 채우지 않는다.
6. 답변의 각 사실 바로 뒤에 근거 문서 번호 [1]처럼 붙인다.
7. HyDE 검색 문장과 모델의 사전 지식은 근거로 사용하지 않는다.

출력 형식:
근거:
- 항목명: [문서 번호] "문서에서 그대로 가져온 짧은 문구"
- 근거가 없는 항목명: 확인 불가

답변:
근거가 확인된 내용만 간결하게 답한다. 근거가 없는 항목은 확인할 수 없다고 밝힌다.

<문서>
{context}
</문서>

<질문>
{query}
</질문>
'''

answers = []
for i, item in enumerate(evaluation_data, start=1):
    query = item["query"]
    search_text, found, reranked_docs = retrieve_and_rerank(query, method)
    context = "\n\n".join(
        f"[{j}] 메타데이터: {doc.metadata}\n본문: {doc.page_content}"
        for j, doc in enumerate(reranked_docs, start=1)
    )
    answer = chat_model.invoke(
        answer_rules.format(context=context, query=query)
    ).content.strip()

    answers.append({
        "query": query,
        "search_text": search_text,
        "answer": answer,
        "contexts": [doc.page_content for doc in reranked_docs],
        "chunk_ids": [doc.metadata["chunk_id"] for doc in reranked_docs],
    })
    print(f"{i}번째 질문: {query}")
    print("사용 청크:", answers[-1]["chunk_ids"])
    print("답변:", answer, "\n")


## 11. RAGAS 평가

위에서 만든 `answers`를 다시 생성하지 않고 평가합니다. `LLMContextRecall`은 참조답의 사실이 검색 문맥에 얼마나 포함되는지, `Faithfulness`는 답변이 검색 문맥에 의해 뒷받침되는지, `FactualCorrectness`는 답변이 참조답과 얼마나 일치하는지 채점합니다. 평가에는 위에서 사용한 로컬 `chat_model`을 사용합니다. 180문항 전체 평가는 시간이 오래 걸릴 수 있습니다.


In [ ]:
import os
import sys
import types

# 현재 환경의 VertexAI import 오류 우회
module_name = "langchain_community.chat_models.vertexai"
if module_name not in sys.modules:
    module = types.ModuleType(module_name)
    module.ChatVertexAI = type("ChatVertexAI", (), {})
    sys.modules[module_name] = module

from langchain_openai import ChatOpenAI
from ragas import EvaluationDataset, evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import Faithfulness, FactualCorrectness
from ragas.run_config import RunConfig

# Colab 보안 비밀에 등록한 경우 환경 변수로 전달
if not os.getenv("OPENAI_API_KEY"):
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

N = 10
if "answers" not in globals() or len(answers) < N:
    raise ValueError("먼저 답변 생성 셀을 실행해 주세요.")

items = evaluation_data[:N]
selected_answers = answers[:N]

if any(item["query"] != answer["query"]
       for item, answer in zip(items, selected_answers)):
    raise ValueError("평가 질문과 생성 답변의 순서가 다릅니다.")

samples = []
for item, answer in zip(items, selected_answers):
    response = answer["answer"].split("답변:", 1)[-1].strip()
    samples.append({
        "user_input": item["query"],
        "retrieved_contexts": answer["contexts"],
        "response": response,
        "reference": item["reference"],
    })

judge = LangchainLLMWrapper(
    ChatOpenAI(model="gpt-5-mini-2025-08-07"),
    bypass_temperature=True,
    bypass_n=True,
)

result = evaluate(
    dataset=EvaluationDataset.from_list(samples),
    metrics=[Faithfulness(), FactualCorrectness()],
    llm=judge,
    run_config=RunConfig(max_workers=1, max_retries=1, timeout=600),
    raise_exceptions=True,
)

df = result.to_pandas()
df.insert(0, "id", [item["id"] for item in items])
df.insert(1, "project", [item["project"] for item in items])
df.insert(2, "question_type", [item["question_type"] for item in items])

score_columns = ["faithfulness", "factual_correctness(mode=f1)"]
display(df[["id", "project", "question_type", *score_columns]])

if df[score_columns].isna().any().any():
    raise RuntimeError("누락된 평가 점수가 있어 비교할 수 없습니다.")

variant = "raw"  # 정제·보존검사 노트북에서는 "clean"
output_path = eval_dataset_path.with_name(f"ragas_eval_{variant}_{N}.csv")
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print("저장:", output_path)